In [1]:
# ============================================================
#  MOTOR SIMBÓLICO PARA L = f(R)
#  Ejemplo: L = R + alpha R^2
# ============================================================

import sympy as sp
from sympy.printing.latex import LatexPrinter

# ============================================================
# SÍMBOLOS BÁSICOS
# ============================================================

R = sp.Symbol("R")
alpha = sp.Symbol("alpha")
Ddim = sp.Symbol("Ddim")

# Índices abstractos
a, b, c, d, i, j, m, n, p, q = sp.symbols("a b c d i j m n p q")

sqrtmg = sp.Symbol("sqrtmg")
GradR2 = sp.Symbol("GradR2")

# ============================================================
# FUNCIONES TENSORIALES ABSTRACTAS
# ============================================================

gU = sp.Function("gU")               # g^{ab}
gD = sp.Function("gD")               # g_{ab}

Ricci = sp.Function("Ricci")         # R_ab

dGU = sp.Function("dGU")             # delta g^{ab}
dGD = sp.Function("dGD")             # delta g_ab

Nabla = sp.Function("Nabla")         # Nabla(a, expr)
NablaU = sp.Function("NablaU")       # Nabla^a(expr)

Box = sp.Function("Box")             # Box(expr)

dGamma = sp.Function("dGamma")       # delta Gamma^a_bc

Delta = sp.Function("Delta")         # delta^a_b

In [2]:
# ============================================================
# DERIVADAS ESCALARES
# ============================================================

def F_from_L(L):
    """
    F(R) = dL/dR
    """
    return sp.diff(L, R)


def grad_scalar(F, idx):
    """
    ∇_idx F(R) = F'(R) ∇_idx R
    """
    return sp.diff(F, R) * Nabla(idx, R)


def grad_up_scalar(F, idx):
    """
    ∇^idx F(R) = F'(R) ∇^idx R
    """
    return sp.diff(F, R) * NablaU(idx, R)


def hess_scalar(F, idx1, idx2):
    """
    ∇_idx1 ∇_idx2 F(R)
    =
    F'(R) ∇_idx1 ∇_idx2 R
    +
    F''(R) ∇_idx1 R ∇_idx2 R
    """
    return (
        sp.diff(F, R) * Nabla(idx1, Nabla(idx2, R))
        + sp.diff(F, R, 2) * Nabla(idx1, R) * Nabla(idx2, R)
    )


def box_scalar(F):
    """
    □F(R)
    =
    F'(R) □R
    +
    F''(R) (∇R)^2
    """
    return (
        sp.diff(F, R) * Box(R)
        + sp.diff(F, R, 2) * GradR2
    )

In [3]:
# ============================================================
# OBJETOS DE LA TEORÍA
# ============================================================

def P_contra(a, b, c, d, L):
    """
    P^{abcd} = ∂L/∂R_abcd
    para L = f(R)
    """
    F = F_from_L(L)
    return sp.simplify(
        sp.Rational(1, 2) * F * (gU(a, c)*gU(b, d) - gU(a, d)*gU(b, c))
    )


def P_mixed(a, m, n, b, L):
    """
    P_a^{m n}_b
    """
    F = F_from_L(L)
    return sp.simplify(
        sp.Rational(1, 2) * F * (Delta(a, n)*Delta(b, m) - gD(a, b)*gU(m, n))
    )


def div_P(b, c, d, L):
    """
    ∇_a P^{abcd}
    """
    F = F_from_L(L)
    return sp.simplify(
        sp.Rational(1, 2) *
        (
            grad_up_scalar(F, c) * gU(b, d)
            - grad_up_scalar(F, d) * gU(b, c)
        )
    )


def div_P_boundary(i, j, d, L):
    """
    ∇_c P^{ijcd}
    usado en δv^j
    """
    F = F_from_L(L)
    return sp.simplify(
        sp.Rational(1, 2) *
        (
            grad_up_scalar(F, i) * gU(j, d)
            - gU(i, d) * grad_up_scalar(F, j)
        )
    )


def Rcal(a, b, L):
    """
    R especial:
    mathcal{R}_{ab} = P_a^{ijk} R_bijk

    Para f(R):
    mathcal{R}_{ab} = F R_ab
    """
    F = F_from_L(L)
    return sp.simplify(F * Ricci(a, b))


def derivative_term(a, b, L):
    """
    -2 ∇_m ∇_n P_a^{m n}_b = g_ab □F - ∇_a∇_b F
    """
    F = F_from_L(L)
    return sp.simplify(
        gD(a, b) * box_scalar(F)
        - hess_scalar(F, a, b)
    )


def E_ab(a, b, L):
    """
    E_ab =
    mathcal{R}_ab - 1/2 g_ab L - 2 ∇_m∇_n P_a^{m n}_b
    """
    return sp.simplify(
        Rcal(a, b, L)
        - sp.Rational(1, 2) * gD(a, b) * L
        + derivative_term(a, b, L)
    )


def trace_E(L, Dim):
    """
    E = g^{ab}E_ab
    para L = f(R)
    """
    F = F_from_L(L)
    return sp.simplify(F * R- sp.Rational(1, 2) * Dim * L+ (Dim - 1) * box_scalar(F))

In [4]:
# ============================================================
# VARIACIÓN DE LA ACCIÓN
# ============================================================

def delta_Ricci(a, b):
    """
    δR_ab = ∇_c δΓ^c_ab - ∇_b δΓ^c_ac
    """
    c_dummy = sp.Symbol("c")
    return (Nabla(c_dummy, dGamma(c_dummy, a, b))- Nabla(b, dGamma(c_dummy, a, c_dummy)))


def raw_variation_density(L):
    """
    Variación antes de integrar por partes:

    δ(√-g L) = √-g [
      (F R_ab - 1/2 g_ab L) δg^{ab} + F g^{ab} δR_ab
    ]
    """
    F = F_from_L(L)
    return sp.simplify(
        sqrtmg *
        (
            (F * Ricci(a, b) - sp.Rational(1, 2) * gD(a, b) * L) * dGU(a, b)
            + F * gU(a, b) * delta_Ricci(a, b)
        )
    )


def delta_v(j, L):
    """
    δv^j =
    2 P^{ibjd} ∇_b δg_di - 2 δg_di ∇_c P^{ijcd}

    Para f(R) se expande automáticamente.
    """
    F = F_from_L(L)

    first_part = (
        F *
        (gU(i, j)*gU(b, d) - gU(i, d)*gU(b, j))
        * Nabla(b, dGD(d, i))
    )

    second_part = (
        -2 * dGD(d, i) * div_P_boundary(i, j, d, L)
    )

    return sp.simplify(first_part + second_part)


def integrated_variation_density(L):
    """
    Variación después de integrar por partes:

    δ(√-g L)
    =
    √-g [
      E_ab δg^{ab} + ∇_j δv^j
    ]
    """
    return sp.simplify(
        sqrtmg *
        (
            E_ab(a, b, L) * dGU(a, b)
            + Nabla(j, delta_v(j, L))
        )
    )


def lovelock_condition_fR(L):
    """
    En el sector f(R), la condición ∇P = 0 equivale a f''(R)=0.
    """
    return sp.simplify(sp.diff(L, R, 2) == 0)

In [5]:
# ============================================================
# FUNCIÓN MAESTRA
# ============================================================

def analyze_fR(L, Dim=Ddim):
    return {
        "Lagrangian L": L,
        "F(R)=dL/dR": F_from_L(L),
        "P^{abcd}": P_contra(a, b, c, d, L),
        "P_a^{m n}_b": P_mixed(a, m, n, b, L),
        "DivP = ∇_a P^{abcd}": div_P(b, c, d, L),
        "Rcal_ab": Rcal(a, b, L),
        "Derivative term": derivative_term(a, b, L),
        "E_ab": E_ab(a, b, L),
        "Trace E": trace_E(L, Dim),
        "Raw variation density": raw_variation_density(L),
        "Boundary vector delta v^j": delta_v(j, L),
        "Integrated variation density": integrated_variation_density(L),
        "Is Lovelock-like in f(R) sector?": lovelock_condition_fR(L)
    }

In [6]:
# ============================================================
# IMPRESIÓN BONITA EN LATEX
# ============================================================

class TensorLatexPrinter(LatexPrinter):
    def _print_Symbol(self, expr):
        name = str(expr)
        if name == "alpha":
            return r"\alpha"
        if name == "Ddim":
            return r"D"
        if name == "sqrtmg":
            return r"\sqrt{-g}"
        if name == "GradR2":
            return r"(\nabla R)^2"
        return super()._print_Symbol(expr)

    def _print_Function(self, expr):
        name = expr.func.__name__
        args = expr.args

        if name == "gU":
            return rf"g^{{{self._print(args[0])}{self._print(args[1])}}}"

        if name == "gD":
            return rf"g_{{{self._print(args[0])}{self._print(args[1])}}}"

        if name == "Ricci":
            return rf"R_{{{self._print(args[0])}{self._print(args[1])}}}"

        if name == "dGU":
            return rf"\delta g^{{{self._print(args[0])}{self._print(args[1])}}}"

        if name == "dGD":
            return rf"\delta g_{{{self._print(args[0])}{self._print(args[1])}}}"

        if name == "Nabla":
            return rf"\nabla_{{{self._print(args[0])}}}\left({self._print(args[1])}\right)"

        if name == "NablaU":
            return rf"\nabla^{{{self._print(args[0])}}}\left({self._print(args[1])}\right)"

        if name == "Box":
            return rf"\Box\left({self._print(args[0])}\right)"

        if name == "dGamma":
            return rf"\delta\Gamma^{{{self._print(args[0])}}}_{{{self._print(args[1])}{self._print(args[2])}}}"

        if name == "Delta":
            return rf"\delta^{{{self._print(args[0])}}}_{{{self._print(args[1])}}}"

        return super()._print_Function(expr)


def latex_tensor(expr):
    printer = TensorLatexPrinter()
    return printer.doprint(expr)


def print_latex(expr):
    print(latex_tensor(expr))


def export_result_to_tex(result, filename="resultado_variacion_fR.tex"):
    keys = [
        "Lagrangian L",
        "F(R)=dL/dR",
        "P^{abcd}",
        "P_a^{m n}_b",
        "DivP = ∇_a P^{abcd}",
        "Rcal_ab",
        "Derivative term",
        "E_ab",
        "Trace E",
        "Raw variation density",
        "Boundary vector delta v^j",
        "Integrated variation density",
        "Is Lovelock-like in f(R) sector?"
    ]

    content = ""

    for key in keys:
        content += rf"\subsection*{{{key}}}" + "\n"
        content += r"\[" + "\n"
        content += latex_tensor(result[key]) + "\n"
        content += r"\]" + "\n\n"

    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)

    return filename

In [13]:
# ============================================================
# EJEMPLO PRINCIPAL
# ============================================================
beta = sp.Symbol("beta")
Lambda = sp.Symbol("Lambda")

Linput = R + alpha*(R**2) - 2*Lambda

result = analyze_fR(Linput, Ddim)

# Mostrar en consola como LaTeX
print("F(R):")
print_latex(result["F(R)=dL/dR"])

print("\nP^{abcd}:")
print_latex(result["P^{abcd}"])

print("\nDivP:")
print_latex(result["DivP = ∇_a P^{abcd}"])

print("\nRcal_ab:")
print_latex(result["Rcal_ab"])

print("\nDerivative term:")
print_latex(result["Derivative term"])

print("\nE_ab:")
print_latex(result["E_ab"])

print("\nTrace E:")
print_latex(result["Trace E"])

print("\nBoundary vector delta v^j:")
print_latex(result["Boundary vector delta v^j"])

# Exportar todo a .tex
filename = export_result_to_tex(result, "variacion_R_alpha_R2.tex")
print(f"\nArchivo exportado: {filename}")

# Traza en D = 4
print("\nTrace E in D=4:")
print_latex(trace_E(Linput, 4))

F(R):
2 R \alpha + 1

P^{abcd}:
\frac{\left(2 R \alpha + 1\right) \left(g^{ac} g^{bd} - g^{ad} g^{bc}\right)}{2}

DivP:
\alpha \left(\nabla^{c}\left(R\right) g^{bd} - \nabla^{d}\left(R\right) g^{bc}\right)

Rcal_ab:
\left(2 R \alpha + 1\right) R_{ab}

Derivative term:
2 \alpha \left(\Box\left(R\right) g_{ab} - \nabla_{a}\left(\nabla_{b}\left(R\right)\right)\right)

E_ab:
2 \alpha \left(\Box\left(R\right) g_{ab} - \nabla_{a}\left(\nabla_{b}\left(R\right)\right)\right) + \left(2 R \alpha + 1\right) R_{ab} - \frac{\left(- 2 \Lambda + R^{2} \alpha + R\right) g_{ab}}{2}

Trace E:
- \frac{D \left(- 2 \Lambda + R^{2} \alpha + R\right)}{2} + R \left(2 R \alpha + 1\right) + 2 \alpha \left(D - 1\right) \Box\left(R\right)

Boundary vector delta v^j:
- 2 \alpha \left(\nabla^{i}\left(R\right) g^{jd} - \nabla^{j}\left(R\right) g^{id}\right) \delta g_{di} + \left(2 R \alpha + 1\right) \left(g^{bd} g^{ij} - g^{bj} g^{id}\right) \nabla_{b}\left(\delta g_{di}\right)

Archivo exportado: variacion_R_alpha

# $L=R$:
$F(R) = \frac{dL}{dR}$:
1

$P^{abcd}$:
$\frac{g^{ac} g^{bd}}{2} - \frac{g^{ad} g^{bc}}{2}$

DivP:
$0$

$Rcal_{ab}$:
$R_{ab}$

Derivative term:
$0$

$E_{ab}$:
$- \frac{R g_{ab}}{2} + R_{ab}$

Trace $E$:
$\frac{R \left(2 - D\right)}{2}$

Boundary vector delta $v^j$:
$\left(g^{bd} g^{ij} - g^{bj} g^{id}\right) \nabla_{b}\left(\delta g_{di}\right)$

# $L = R + \alpha R^2$

$F(R) = \frac{dL}{dR}$:
$2 R \alpha + 1$

$P^{abcd}$:
$\frac{\left(2 R \alpha + 1\right) \left(g^{ac} g^{bd} - g^{ad} g^{bc}\right)}{2}$

DivP:
$\alpha \left(\nabla^{c}\left(R\right) g^{bd} - \nabla^{d}\left(R\right) g^{bc}\right)$

$Rcal_{ab}$:
$\left(2 R \alpha + 1\right) R_{ab}$

Derivative term:
$2 \alpha \left(\Box\left(R\right) g_{ab} - \nabla_{a}\left(\nabla_{b}\left(R\right)\right)\right)$

$E_{ab}$:
$- \frac{R \left(R \alpha + 1\right) g_{ab}}{2} + 2 \alpha \left(\Box\left(R\right) g_{ab} - \nabla_{a}\left(\nabla_{b}\left(R\right)\right)\right) + \left(2 R \alpha + 1\right) R_{ab}$

Trace $E$:
$- \frac{D R \left(R \alpha + 1\right)}{2} + R \left(2 R \alpha + 1\right) + 2 \alpha \left(D - 1\right) \Box\left(R\right)$

Boundary vector delta $v^j$:
$- 2 \alpha \left(\nabla^{i}\left(R\right) g^{jd} - \nabla^{j}\left(R\right) g^{id}\right) \delta g_{di} + \left(2 R \alpha + 1\right) \left(g^{bd} g^{ij} - g^{bj} g^{id}\right) \nabla_{b}\left(\delta g_{di}\right)$

Archivo exportado: variacion_R_alpha_R2.tex

Trace $E$ in $D=4$:
$- R + 6 \alpha \Box\left(R\right)$

# $L = R + \alpha R^2 + \beta R^3$

$F(R) = \frac{dL}{dR}$:
$3 R^{2} \beta + 2 R \alpha + 1$

$P^{abcd}$:
$\frac{\left(g^{ac} g^{bd} - g^{ad} g^{bc}\right) \left(3 R^{2} \beta + 2 R \alpha + 1\right)}{2}$

DivP:
$\left(3 R \beta + \alpha\right) \left(\nabla^{c}\left(R\right) g^{bd} - \nabla^{d}\left(R\right) g^{bc}\right)$

$Rcal_{ab}$:
$\left(3 R^{2} \beta + 2 R \alpha + 1\right) R_{ab}$

Derivative term:
$- 6 \beta \nabla_{a}\left(R\right) \nabla_{b}\left(R\right) + 2 \left(3 (\nabla R)^2 \beta + \left(3 R \beta + \alpha\right) \Box\left(R\right)\right) g_{ab} - 2 \left(3 R \beta + \alpha\right) \nabla_{a}\left(\nabla_{b}\left(R\right)\right)$

$E_{ab}$:
$- \frac{R \left(R^{2} \beta + R \alpha + 1\right) g_{ab}}{2} - 6 \beta \nabla_{a}\left(R\right) \nabla_{b}\left(R\right) + 2 \left(3 (\nabla R)^2 \beta + \left(3 R \beta + \alpha\right) \Box\left(R\right)\right) g_{ab} - 2 \left(3 R \beta + \alpha\right) \nabla_{a}\left(\nabla_{b}\left(R\right)\right) + \left(3 R^{2} \beta + 2 R \alpha + 1\right) R_{ab}$

Trace $E$:
$- \frac{D R \left(R^{2} \beta + R \alpha + 1\right)}{2} + R \left(3 R^{2} \beta + 2 R \alpha + 1\right) + 2 \left(D - 1\right) \left(3 (\nabla R)^2 \beta + \left(3 R \beta + \alpha\right) \Box\left(R\right)\right)$

Boundary vector delta $v^j$:
$- 2 \left(3 R \beta + \alpha\right) \left(\nabla^{i}\left(R\right) g^{jd} - \nabla^{j}\left(R\right) g^{id}\right) \delta g_{di} + \left(g^{bd} g^{ij} - g^{bj} g^{id}\right) \left(3 R^{2} \beta + 2 R \alpha + 1\right) \nabla_{b}\left(\delta g_{di}\right)$

Archivo exportado: variacion_R_alpha_R2.tex

Trace $E$ in $D=4$:

$18 (\nabla R)^2 \beta + R^{3} \beta + 18 R \beta \Box\left(R\right) - R + 6 \alpha \Box\left(R\right)$

# $L = R + \alpha R^2 + \beta R^3 + \sin(R) + e^R$

$F(R)$:
$3 R^{2} \beta + 2 R \alpha + e^{R} + \cos{\left(R \right)} + 1$

$P^{abcd}$:
$\frac{\left(g^{ac} g^{bd} - g^{ad} g^{bc}\right) \left(3 R^{2} \beta + 2 R \alpha + e^{R} + \cos{\left(R \right)} + 1\right)}{2}$

DivP:
$\left(\nabla^{c}\left(R\right) g^{bd} - \nabla^{d}\left(R\right) g^{bc}\right) \left(3 R \beta + \alpha + \frac{e^{R}}{2} - \frac{\sin{\left(R \right)}}{2}\right)$

$Rcal_{ab}$:
$\left(3 R^{2} \beta + 2 R \alpha + e^{R} + \cos{\left(R \right)} + 1\right) R_{ab}$

Derivative term:
$\left((\nabla R)^2 \left(6 \beta + e^{R} - \cos{\left(R \right)}\right) + \left(6 R \beta + 2 \alpha + e^{R} - \sin{\left(R \right)}\right) \Box\left(R\right)\right) g_{ab} - \left(6 \beta + e^{R} - \cos{\left(R \right)}\right) \nabla_{a}\left(R\right) \nabla_{b}\left(R\right) - \left(6 R \beta + 2 \alpha + e^{R} - \sin{\left(R \right)}\right) \nabla_{a}\left(\nabla_{b}\left(R\right)\right)$

$E_{ab}$:
$\left((\nabla R)^2 \left(6 \beta + e^{R} - \cos{\left(R \right)}\right) + \left(6 R \beta + 2 \alpha + e^{R} - \sin{\left(R \right)}\right) \Box\left(R\right)\right) g_{ab} - \left(6 \beta + e^{R} - \cos{\left(R \right)}\right) \nabla_{a}\left(R\right) \nabla_{b}\left(R\right) - \left(6 R \beta + 2 \alpha + e^{R} - \sin{\left(R \right)}\right) \nabla_{a}\left(\nabla_{b}\left(R\right)\right) + \left(3 R^{2} \beta + 2 R \alpha + e^{R} + \cos{\left(R \right)} + 1\right) R_{ab} - \frac{\left(R^{3} \beta + R^{2} \alpha + R + e^{R} + \sin{\left(R \right)}\right) g_{ab}}{2}$

Trace $E$:
$- \frac{D \left(R^{3} \beta + R^{2} \alpha + R + e^{R} + \sin{\left(R \right)}\right)}{2} + R \left(3 R^{2} \beta + 2 R \alpha + e^{R} + \cos{\left(R \right)} + 1\right) + \left(D - 1\right) \left((\nabla R)^2 \left(6 \beta + e^{R} - \cos{\left(R \right)}\right) + \left(6 R \beta + 2 \alpha + e^{R} - \sin{\left(R \right)}\right) \Box\left(R\right)\right)$

Boundary vector delta $v^j$:
$- \left(\nabla^{i}\left(R\right) g^{jd} - \nabla^{j}\left(R\right) g^{id}\right) \left(6 R \beta + 2 \alpha + e^{R} - \sin{\left(R \right)}\right) \delta g_{di} + \left(g^{bd} g^{ij} - g^{bj} g^{id}\right) \left(3 R^{2} \beta + 2 R \alpha + e^{R} + \cos{\left(R \right)} + 1\right) \nabla_{b}\left(\delta g_{di}\right)$

Archivo exportado: variacion_R_alpha_R2.tex

Trace $E$ in $D=4$:
$18 (\nabla R)^2 \beta + 3 (\nabla R)^2 e^{R} - 3 (\nabla R)^2 \cos{\left(R \right)} + R^{3} \beta + 18 R \beta \Box\left(R\right) + R e^{R} + R \cos{\left(R \right)} - R + 6 \alpha \Box\left(R\right) + 3 \Box\left(R\right) e^{R} - 3 \Box\left(R\right) \sin{\left(R \right)} - 2 e^{R} - 2 \sin{\left(R \right)}$

# $L = R + \alpha R^2 - 2\Lambda$

f(R) = $2 R \alpha + 1$

P^{abcd}:
$\frac{\left(2 R \alpha + 1\right) \left(g^{ac} g^{bd} - g^{ad} g^{bc}\right)}{2}$

DivP:
$\alpha \left(\nabla^{c}\left(R\right) g^{bd} - \nabla^{d}\left(R\right) g^{bc}\right)$

Rcal_ab:
$\left(2 R \alpha + 1\right) R_{ab}$

Derivative term:
$2 \alpha \left(\Box\left(R\right) g_{ab} - \nabla_{a}\left(\nabla_{b}\left(R\right)\right)\right)$

E_ab:
$2 \alpha \left(\Box\left(R\right) g_{ab} - \nabla_{a}\left(\nabla_{b}\left(R\right)\right)\right) + \left(2 R \alpha + 1\right) R_{ab} - \frac{\left(- 2 \Lambda + R^{2} \alpha + R\right) g_{ab}}{2}$

Trace E:
$- \frac{D \left(R^{2} \alpha + R - 2 \Lambda\right)}{2} + R \left(2 R \alpha + 1\right) + 2 \alpha \left(D - 1\right) \Box\left(R\right)$

Boundary vector delta v^j:
$- 2 \alpha \left(\nabla^{i}\left(R\right) g^{jd} - \nabla^{j}\left(R\right) g^{id}\right) \delta g_{di} + \left(2 R \alpha + 1\right) \left(g^{bd} g^{ij} - g^{bj} g^{id}\right) \nabla_{b}\left(\delta g_{di}\right)$

Archivo exportado: variacion_R_alpha_R2.tex

Trace E in D=4:
$- R + 6 \alpha \Box\left(R\right) + 4 \Lambda$